In [6]:
# process requirement script
# replace raw cookie and csrf token with a current valid cookie

import requests
import json

BASE_URL = "http://localhost:82"
PREFIX = "/bcap"

# Your active credentials
RAW_COOKIE = 'username-localhost-8888="2|1:0|10:1778013652|23:username-localhost-8888|196:eyJ1c2VybmFtZSI6ICI2YzczYThmNGFjY2E0YjIzOTFkZDg4NmY1Njg2YTRkOCIsICJuYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiZGlzcGxheV9uYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiaW5pdGlhbHMiOiAiQVQiLCAiY29sb3IiOiBudWxsfQ==|18fab2f76018c29028bfeef14fcccb33aa5453d4fe73bf39f039b6497ba0700a"; _xsrf=2|7319bcc9|b85949285c2c715d7addfcdbeb20bbbb|1778013652; csrftoken=8eTXQ39niTUFgXbIIIQBkTupc8rdxTmH; bcap_dev=wpp5ck1j3mj6v2ee0yemu8i35q918lta'
RAW_CSRF_TOKEN = "8eTXQ39niTUFgXbIIIQBkTupc8rdxTmH"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json", 
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": f"{BASE_URL}{PREFIX}/",
    "Cookie": RAW_COOKIE,
    "X-CSRFToken": RAW_CSRF_TOKEN
}

# 2. The exact base URL (No UUID, No trailing slash)
post_url = f"{BASE_URL}{PREFIX}/api/resource/process_requirement?format=json"

print(f"Targeting URL: {post_url}")

# 3. Build the payload (Now with the mandatory Requirement Identification field!)
payload = {
    "resourceinstanceid": None,
    "aliased_data": {
        "requirement_identification": {
            "tileid": None,
            "aliased_data": {
                # THIS IS THE MISSING FIELD THE SERVER WAS ASKING FOR
                "requirement_identification": {
                    "node_value": "REQ-001", 
                    "display_value": "",
                    "details": []
                },
                "requirement_name": {
                    "node_value": "Jupyter Raw Cookie Requirement",
                    "display_value": "",
                    "details": []
                }
            }
        },
        "sub_requirement": [
            {
                "tileid": None,
                "aliased_data": {
                    "sub_requirement_sort_order": {
                        "node_value": 1,
                        "display_value": "",
                        "details": []
                    },
                    "sub_requirement_name": {
                        "node_value": "Step 1: Test the clean POST",
                        "display_value": "",
                        "details": []
                    },
                    "sub_requirement_description": {
                        "node_value": "We successfully let the server generate its own UUID!",
                        "display_value": "",
                        "details": []
                    }
                }
            }
        ]
    }
}

# 4. Fire the POST request
print("Submitting new record to database using POST...")
response = requests.post(post_url, headers=headers, data=json.dumps(payload), allow_redirects=False)

print(f"\n--- RESULTS ---")
print(f"Status Code: {response.status_code}")

if response.status_code in [200, 201]:
    try:
        saved_data = response.json()
        print("✅ ULTIMATE SUCCESS! The database accepted the record.")
        
        # Extract the ID the server generated for us to build the link
        new_id = saved_data.get("resourceinstanceid")
        if new_id:
            print(f"The server assigned UUID: {new_id}")
            print(f"Go check your UI at: {BASE_URL}{PREFIX}/report/{new_id}")
        else:
            print("Record saved, but could not parse the new UUID from the response.")
            
    except json.JSONDecodeError:
        print("🚨 The server accepted the POST (200 OK), but returned HTML instead of the expected JSON.")
else:
    print("🚨 Something went wrong on the save.")
    print(response.text[:500])

Targeting URL: http://localhost:82/bcap/api/resource/process_requirement?format=json
Submitting new record to database using POST...

--- RESULTS ---
Status Code: 302
🚨 Something went wrong on the save.



In [6]:
# permit application script
# creates permit application and links a process requirement
# replace raw cookie and csrf token with a current valid cookie

import requests
import json

BASE_URL = "http://localhost:82"
PREFIX = "/bcap"

# ==========================================
# 1. AUTHENTICATION & CONFIGURATION
# ==========================================
RAW_COOKIE = 'username-localhost-8888="2|1:0|10:1778013652|23:username-localhost-8888|196:eyJ1c2VybmFtZSI6ICI2YzczYThmNGFjY2E0YjIzOTFkZDg4NmY1Njg2YTRkOCIsICJuYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiZGlzcGxheV9uYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiaW5pdGlhbHMiOiAiQVQiLCAiY29sb3IiOiBudWxsfQ==|18fab2f76018c29028bfeef14fcccb33aa5453d4fe73bf39f039b6497ba0700a"; _xsrf=2|7319bcc9|b85949285c2c715d7addfcdbeb20bbbb|1778013652; csrftoken=0lNrsLwyRQte5NjyMVd77u8sDl1MqbY5; bcap_dev=k8urpyougkjrhs3o1dzfvx4ktqvxrrw2'
RAW_CSRF_TOKEN = "0lNrsLwyRQte5NjyMVd77u8sDl1MqbY5"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json", 
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": f"{BASE_URL}{PREFIX}/",
    "Cookie": RAW_COOKIE,
    "X-CSRFToken": RAW_CSRF_TOKEN
}

# ==========================================
# Phase 1: CREATE PROCESS REQUIREMENT
# ==========================================
print("Phase 1: Generating Process Requirement with 5 sub-steps...")

req_post_url = f"{BASE_URL}{PREFIX}/api/resource/process_requirement?format=json"

sub_requirements_list = []
steps = [
    ("Step 1: Initial Archaeological Consultation", "Consult with BC Archaeological Branch and local First Nations to identify site potential."),
    ("Step 2: Preliminary Field Survey & Mapping", "Conduct an on-site pedestrian field survey and map known archaeological features."),
    ("Step 3: Archaeological Impact Assessment", "Analyze potential developmental impacts on any identified cultural heritage sites."),
    ("Step 4: Mitigative Design Proposal", "Draft a comprehensive mitigative project design to minimize impact on heritage assets."),
    ("Step 5: Final Inspection & Report Submission", "Perform a post-development inspection and submit final paperwork for regulatory sign-off.")
]

for idx, (name, desc) in enumerate(steps, start=1):
    sub_requirements_list.append({
        "tileid": None,
        "aliased_data": {
            "sub_requirement_sort_order": {
                "node_value": idx,
                "display_value": "",
                "details": []
            },
            "sub_requirement_name": {
                "node_value": name,
                "display_value": "",
                "details": []
            },
            "sub_requirement_description": {
                "node_value": desc,
                "display_value": "",
                "details": []
            }
        }
    })

req_payload = {
    "resourceinstanceid": None,
    "aliased_data": {
        "requirement_identification": {
            "tileid": None,
            "aliased_data": {
                "requirement_identification": {
                    "node_value": "REQ-AUTO-001", 
                    "display_value": "",
                    "details": []
                },
                "requirement_name": {
                    "node_value": "Heritage Clearance Checklist",
                    "display_value": "",
                    "details": []
                }
            }
        },
        "sub_requirement": sub_requirements_list
    }
}

req_response = requests.post(req_post_url, headers=headers, data=json.dumps(req_payload), allow_redirects=False)

if req_response.status_code not in [200, 201]:
    print(f"🚨 Phase 1 Failed! Status: {req_response.status_code}")
    print(req_response.text[:500])
    raise Exception("Stopping script. Process Requirement creation failed.")

new_req_id = req_response.json().get("resourceinstanceid")
print(f"✅ Process Requirement Created! Assigned UUID: {new_req_id}")

# ==========================================
# Phase 2: CREATE PERMIT APP & LINK IT
# ==========================================
print("\nPhase 2: Creating Permit Application and linking to Requirement...")

app_post_url = f"{BASE_URL}{PREFIX}/api/resource/permit_application?format=json"

app_payload = {
    "resourceinstanceid": None,
    "aliased_data": {
        "application_identification": {
            "tileid": None,
            "aliased_data": {
                "application_id": {
                    "node_value": "APP-AUTO-002", 
                    "display_value": "",
                    "details": []
                },
                "project_name": {
                    "node_value": "End-to-End Linked Pipeline Test (Lean)", 
                    "display_value": "",
                    "details": []
                }
            }
        },
        "application_admin": {
            "tileid": None,
            "aliased_data": {
                "process_requirement": [
                    {
                        "tileid": None,
                        "aliased_data": {
                            "process_requirement": {
                                "node_value": [{"resourceId": new_req_id}], 
                                "display_value": "",
                                "details": []
                            },
                            "process_requirement_order": {
                                "node_value": 1,
                                "display_value": "",
                                "details": []
                            }
                        }
                    }
                ]
            }
        }
    } 
} 

app_response = requests.post(app_post_url, headers=headers, data=json.dumps(app_payload), allow_redirects=False)

if app_response.status_code not in [200, 201]:
    print(f"🚨 Phase 2 Failed! Status: {app_response.status_code}")
    print(app_response.text[:500])
    raise Exception("Stopping script. Permit Application creation failed.")

new_app_id = app_response.json().get("resourceinstanceid")

print(f"✅ Permit Application Created! Assigned UUID: {new_app_id}")

# ==========================================
# FINAL RESULTS
# ==========================================
print(f"\n🎉 END-TO-END PIPELINE COMPLETE!")
print("-" * 50)
print(f"View Process Requirement: {BASE_URL}{PREFIX}/report/{new_req_id}")
print(f"View Permit Application:  {BASE_URL}{PREFIX}/report/{new_app_id}")

Phase 1: Generating Process Requirement with 5 sub-steps...
✅ Process Requirement Created! Assigned UUID: 99bba0a9-49db-4017-8414-459fb09cd1a8

Phase 2: Creating Permit Application and linking to Requirement...
✅ Permit Application Created! Assigned UUID: ddfe1ef7-a947-4050-9595-ab868d1cb5c9

🎉 END-TO-END PIPELINE COMPLETE!
--------------------------------------------------
View Process Requirement: http://localhost:82/bcap/report/99bba0a9-49db-4017-8414-459fb09cd1a8
View Permit Application:  http://localhost:82/bcap/report/ddfe1ef7-a947-4050-9595-ab868d1cb5c9
